# groupby and aggregation

Summarise data by group with groupby: totals, averages and counts per product, month or department, several measures at once with named aggregation, and each group's share of the total.

*From [CloudTech Academy](https://academy.cloudtechanalytics.com/learn/python-for-data-analytics/groupby-and-aggregation). Run the cells in order (Runtime → Run all), edit them, and experiment. Answer the practice questions on the lesson page.*

## The problem

Kolanut's managing director asks the questions every manager asks:

> "Which products bring in the most money? How did each month go? And in HR: which departments are we losing people from, and what do we pay at each level?"

Every one of those is **split, apply, combine**: split the rows into groups (by product, month or department), apply a calculation to each group (sum, average, count), and combine the results into one small table. In Excel that's a pivot table. In pandas it's `groupby`, and it's the single most useful tool in this course.

## The concept

### The pattern

In [ ]:
orders.groupby("product_id")["revenue"].sum()

*Expected output:*

```
product_id
1     59804940.0
2     15375160.0
3     74837880.0
4     74594380.0
5     34168995.0
6     32751300.0
7     21029160.0
8     37727020.0
9     78372810.0
10    43396020.0
11    70241610.0
12    52758600.0
13    71586900.0
14    63334275.0
15    44217180.0
16    56345015.0
Name: revenue, dtype: float64
```

Read it left to right: take `orders`, **group by** `product_id`, take the `revenue` column, and **sum** it within each group. The result is a Series with one row per product.

Common calculations: `.sum()`, `.mean()`, `.median()`, `.min()`, `.max()`, `.count()` (non-empty values), `.nunique()` (distinct values) and `.size()` (rows per group, including blanks).

### Sorting and the top N

Add `.sort_values(ascending=False)` to rank the groups, and `.head(5)` for the top five. `.idxmax()` gives the label of the largest group directly.

### Several measures at once: named aggregation

In [ ]:
orders.groupby("product_id").agg(
    revenue=("revenue", "sum"),
    lines=("order_id", "count"),
    avg_quantity=("quantity", "mean"),
)

*Expected output:*

```
revenue  lines  avg_quantity
product_id
1           59804940.0    308     14.538961
2           15375160.0    323     13.120743
3           74837880.0    282     14.326241
4           74594380.0    322     13.723602
5           34168995.0    269     14.111524
6           32751300.0    283     13.837456
7           21029160.0    246     14.256098
8           37727020.0    269     13.308550
9           78372810.0    258     14.244186
10          43396020.0    226     13.491150
11          70241610.0    272     13.580882
12          52758600.0    250     13.596000
13          71586900.0    237     13.333333
14          63334275.0    253     13.237154
15          44217180.0    224     13.745536
16          56345015.0    244     13.844262
```

Each line is `new_name=(column, calculation)`. You get a tidy table with exactly the columns you named.

### Grouping by more than one column

`orders.groupby(["year", "discount_pct"])["revenue"].sum()` gives one row per combination. Add `.unstack()` to turn the second level into columns, which reads like a pivot table.

### Back to an ordinary table: `reset_index()`

The group labels become the result's **index**. `.reset_index()` turns them back into a normal column, which you'll want before merging, charting or saving.

### Shares of a total

Divide each group by the total: `by_product / by_product.sum()`. To put each row's group total back on every row, use `transform`:

In [ ]:
orders["product_total"] = orders.groupby("product_id")["revenue"].transform("sum")
orders[["product_id", "revenue", "product_total"]].head(3)

*Expected output:*

```
product_id   revenue  product_total
0           3  260400.0     74837880.0
1           1   92400.0     59804940.0
2           2   14400.0     15375160.0
```

That's useful for "what share of its product's revenue is this line?"

> **Note:** > `count()` and `size()` differ only when there are blanks: `count()` skips them, `size()` doesn't. When counting rows, `size()` is the safer habit.

## Example

Set up revenue and dates as in lesson 5, then rank products and look at the months:

In [ ]:
import pandas as pd

base = "https://academy.cloudtechanalytics.com/datasets/sales/"
orders = pd.read_csv(base + "orders.csv")
products = pd.read_csv(base + "products.csv")
orders["revenue"] = orders["quantity"] * orders["unit_price"] * (1 - orders["discount_pct"] / 100)
orders["order_date"] = pd.to_datetime(orders["order_date"])
orders["month"] = orders["order_date"].dt.to_period("M")

by_product = orders.groupby("product_id").agg(
    revenue=("revenue", "sum"),
    lines=("order_id", "count"),
    avg_quantity=("quantity", "mean"),
).sort_values("revenue", ascending=False)
by_product.head().round(1)

*Expected output:*

```
revenue  lines  avg_quantity
product_id
9           78372810.0    258          14.2
3           74837880.0    282          14.3
4           74594380.0    322          13.7
13          71586900.0    237          13.3
11          70241610.0    272          13.6
```

Product IDs aren't very readable. A dictionary of names, used with `.map()` (a lookup, like lesson 2's dictionaries), fixes that:

In [ ]:
names = dict(zip(products["product_id"], products["product_name"]))
by_product.index = by_product.index.map(names)
by_product.head(3)

Now the monthly picture:

In [ ]:
monthly = orders.groupby("month")["revenue"].sum()
print(f"Best month: {monthly.idxmax()} (₦{monthly.max():,.0f})")
print(f"Worst month: {monthly.idxmin()} (₦{monthly.min():,.0f})")

*Expected output:*

```
Best month: 2025-12 (₦66,284,310)
Worst month: 2025-02 (₦36,138,690)
```

December is Kolanut's peak: festive stock-up by its retail customers.

## Walkthrough

1. Run the Example's set-up cell and the product ranking. Detergent (product 9) leads, though bottled water (product 2) appears on the most lines: frequent isn't the same as valuable.
2. Run the `names` cell and look at `by_product.head(3)` again with product names.
3. Shares: `share = by_product["revenue"] / by_product["revenue"].sum()` and `share.head(3)`. The top three products bring in about 27% of revenue.
4. Group by two columns: `orders.groupby([orders["order_date"].dt.year, "discount_pct"])["revenue"].sum().unstack()`. Read across a row: one year's revenue split by discount level.
5. Switch to the HR data:

In [ ]:
employees = pd.read_csv("https://academy.cloudtechanalytics.com/datasets/hr/employees.csv")
by_dept = employees.groupby("department").agg(
    staff=("employee_id", "size"),
    resigned=("status", lambda s: (s == "Resigned").sum()),
    avg_salary=("monthly_salary", "mean"),
)
by_dept["resign_rate"] = by_dept["resigned"] / by_dept["staff"]
by_dept.sort_values("resign_rate", ascending=False).round(2)

6. The `lambda` is a small function written in place: for each department's `status` values, it counts how many are `"Resigned"`. Use it when the calculation you need isn't one of the built-in names.
7. Add `.reset_index()` to `by_dept` and see `department` become an ordinary column again.

> **In the business:** > A resignation **rate** beats a resignation **count**. A big department will always lose more people; the rate says whether it's losing a bigger **share** of them.

## Practice

**Practice:** Which **month** had Kolanut's highest number of **order lines** (not revenue)? Answer as YYYY-MM, for example 2025-03.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** Which **customer_id** brought in the most revenue across the whole period?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** In the HR data, which **department** has the highest **resignation rate** (resigned ÷ all staff)?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Challenge

**Practice:** What is the **average monthly salary** of **Senior** staff in the **IT** department? Group by two columns. Round to the nearest naira.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## More practice

Optional drills on the legal dataset. Load `matters.csv` and `invoices.csv` from `https://academy.cloudtechanalytics.com/datasets/legal/`.

**Practice:** Which **responsible_lawyer** handles the most matters?

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

**Practice:** Group the invoices by **status**. What is the **average** invoice amount for **Overdue** invoices? Round to the nearest naira.

*Work it out below, then enter your answer on the lesson page.*

In [ ]:
# Your code here

## Check your understanding